In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib # <--- PERBAIKAN: Import yang hilang
warnings.filterwarnings("ignore")

# Import dari Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, f1_score
)
from sklearn.ensemble import IsolationForest

# Setup Device (CUDA atau CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Menggunakan device: {device} ({torch.cuda.get_device_name(0)})")
else:
    print(f"Menggunakan device: {device}.")

DATA_PATH = r'C:\FRAUD-Sinta2\Output'
MODEL_PATH = r'C:\FRAUD-Sinta2\Output\Output2'
os.makedirs(MODEL_PATH, exist_ok=True)

# Konstanta Model
RANDOM_SEED = 42
TIME_STEPS = 10 # Panjang sekuens untuk LSTM
BATCH_SIZE = 128
N_EPOCHS = 30 # Jumlah epoch (bisa dinaikkan ke 50-100 untuk hasil lebih baik)

# Set random seed untuk reproduktibilitas
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("Libraries imported and setup complete.")

Menggunakan device: cuda (NVIDIA GeForce RTX 3050)
Libraries imported and setup complete.


In [2]:
try:
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'train_v3.csv'))
    df_val = pd.read_csv(os.path.join(DATA_PATH, 'val_v3.csv'))
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test_v3.csv'))
    print("Dataset V3 (Train, Val, Test) berhasil di-load.")
    print(f"Train shape: {df_train.shape}")
    print(f"Val shape: {df_val.shape}")
    print(f"Test shape: {df_test.shape}")
except FileNotFoundError:
    print(f"Error: Pastikan file 'train_v3.csv', 'val_v3.csv', 'test_v3.csv' ada di {DATA_PATH}")
    raise

Dataset V3 (Train, Val, Test) berhasil di-load.
Train shape: (72323, 6)
Val shape: (15750, 6)
Test shape: (15790, 6)


In [3]:
def preprocess_dataframe(df):
    df = df.copy() # Hindari SettingWithCopyWarning
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    return df

print("Preprocessing fitur waktu...")
df_train = preprocess_dataframe(df_train)
df_val = preprocess_dataframe(df_val)
df_test = preprocess_dataframe(df_test)

# --- (PERBAIKAN: 'ValueError: previously unseen labels') ---
# 3. Fit LabelEncoder 'user_id' PADA SEMUA DATA
print("Fitting LabelEncoder...")
# Gabungkan 'user_id' dari semua set untuk 'belajar'
all_users = pd.concat([df_train['user_id'], df_val['user_id'], df_test['user_id']]).unique()
print(f"Total {len(all_users)} user unik ditemukan di semua set.")

le = LabelEncoder()
le.fit(all_users) # 'Fit' pada seluruh "alam semesta" user

# 4. SEKARANG 'Transform' akan berhasil di semua set
df_train['user_id_encoded'] = le.transform(df_train['user_id'])
df_val['user_id_encoded'] = le.transform(df_val['user_id'])
df_test['user_id_encoded'] = le.transform(df_test['user_id'])
print("LabelEncoder di-fit pada SEMUA user, di-transform ke Train/Val/Test.")
# --- (PERBAIKAN SELESAI) ---

# 5. Scaling 'amount' (HANYA fit di Train)
scaler = StandardScaler()
# (Kita fit HANYA pada 'amount' dari train set)
df_train['amount_scaled'] = scaler.fit_transform(df_train[['amount']])
# (Kita transform Val dan Test set)
df_val['amount_scaled'] = scaler.transform(df_val[['amount']])
df_test['amount_scaled'] = scaler.transform(df_test[['amount']])
print("StandardScaler di-fit pada Train, di-transform ke Val/Test.")

# 6. Simpan scaler & encoder (untuk reproduksi)
joblib.dump(le, os.path.join(MODEL_PATH, 'user_id_encoder_sim.joblib'))
joblib.dump(scaler, os.path.join(MODEL_PATH, 'amount_scaler_sim.joblib'))

# --- (PERBAIKAN: 'NameError: df_train_normal not defined') ---
# 7. Definisikan DataFrame Model Siap Pakai
# Ini adalah DataFrame lengkap (termasuk Class/anom_type)
# yang akan digunakan oleh Dataset Class kita nanti.
print("\nMembuat DataFrame siap pakai untuk model...")

FEATURE_COLS = ['user_id_encoded', 'hour_of_day', 'day_of_week', 'amount_scaled']
LABEL_COLS = ['Class', 'anomaly_type']

df_train_model = df_train[FEATURE_COLS + LABEL_COLS].copy()
df_val_model = df_val[FEATURE_COLS + LABEL_COLS].copy()
df_test_model = df_test[FEATURE_COLS + LABEL_COLS].copy()

# Pisahkan data normal untuk melatih AE (INI YANG MEMPERBAIKI NameError)
df_train_normal = df_train_model[df_train_model['Class'] == 0].copy()
# --- (PERBAIKAN SELESAI) ---

# 8. Pisahkan data untuk baseline (NumPy arrays)
X_train_statis = df_train[FEATURE_COLS].values
y_train_statis = df_train['Class'].values
X_test_statis = df_test[FEATURE_COLS].values
y_test_statis = df_test['Class'].values

print(f"Data Normal untuk Training AE: {df_train_normal.shape}")
print(f"Data Full untuk Validasi: {df_val_model.shape}")
print(f"Data Full untuk Test: {df_test_model.shape}")


Preprocessing fitur waktu...
Fitting LabelEncoder...
Total 99437 user unik ditemukan di semua set.
LabelEncoder di-fit pada SEMUA user, di-transform ke Train/Val/Test.
StandardScaler di-fit pada Train, di-transform ke Val/Test.

Membuat DataFrame siap pakai untuk model...
Data Normal untuk Training AE: (72323, 6)
Data Full untuk Validasi: (15750, 6)
Data Full untuk Test: (15790, 6)


In [4]:
# --- Baseline 1: Isolation Forest (Data Statis) ---

print("\n--- [Baseline 1] Melatih Isolation Forest (Simulasi V3) ---")

# --- (PERBAIKAN: 'AttributeError: value_counts') ---
# 'y_train_statis' adalah NumPy array. Kita gunakan 'np.mean()'
contamination_rate = np.mean(y_train_statis)
# --- (PERBAIKAN SELESAI) ---

# Karena df_train 100% normal, rate-nya 0.0. Kita set ke 'auto'.
if contamination_rate == 0.0:
    contamination_rate = 'auto' 
    print("Contamination rate di-set ke 'auto' (karena Train Set 100% normal).")
else:
    print(f"Contamination rate di-set ke: {contamination_rate:.4f}")

model_if = IsolationForest(
    n_estimators=100,
    contamination=contamination_rate, # Gunakan rate yang sudah diperbaiki
    random_state=RANDOM_SEED, 
    n_jobs=-1
)

print("Melatih IF pada data training (statis)...")
# Latih pada data normal (seperti AE)
X_train_normal_statis = X_train_statis[y_train_statis == 0]
model_if.fit(X_train_normal_statis) 

print("Mengevaluasi IF pada data tes...")
y_pred_if_raw = model_if.predict(X_test_statis)
# Ubah skor IF (-1 = anomali, 1 = normal) menjadi label (1 = anomali, 0 = normal)
y_pred_if = [1 if p == -1 else 0 for p in y_pred_if_raw]

print("\n--- Hasil Isolation Forest (Tabel 2) ---")
print(classification_report(y_test_statis, y_pred_if, target_names=['Normal (0)', 'Fraud (1)']))



--- [Baseline 1] Melatih Isolation Forest (Simulasi V3) ---
Contamination rate di-set ke 'auto' (karena Train Set 100% normal).
Melatih IF pada data training (statis)...
Mengevaluasi IF pada data tes...

--- Hasil Isolation Forest (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       1.00      0.65      0.79     15490
   Fraud (1)       0.05      0.98      0.10       300

    accuracy                           0.65     15790
   macro avg       0.53      0.81      0.44     15790
weighted avg       0.98      0.65      0.77     15790



In [5]:
# --- Definisi Class Dataset & Model ---

# (A) Dataset Statis (untuk AE Statis)
class StaticDataset(Dataset):
    def __init__(self, df):
        # Ambil HANYA kolom fitur
        self.features = df.drop(['Class', 'anomaly_type'], axis=1, errors='ignore').values
        self.labels = df['Class'].values

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        # 
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# (B) Arsitektur AE Statis
class Static_Autoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=10):
        super(Static_Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 20), nn.ReLU(),
            nn.Linear(20, embedding_dim), nn.ReLU() # Bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 20), nn.ReLU(),
            nn.Linear(20, n_features)
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# (C) Dataset Sekuensial (untuk LSTM-AE)
class FraudDatasetSeq(Dataset):
    def __init__(self, df, seq_len=10):
        self.df = df
        self.seq_len = seq_len
        
        feature_cols = ['user_id_encoded', 'hour_of_day', 'day_of_week', 'amount_scaled']
        label_col = 'Class'

        self.data_by_user = [
            group[feature_cols].values 
            for _, group in self.df.groupby('user_id_encoded')
        ]
        self.labels_by_user = [
            group[label_col].values
            for _, group in self.df.groupby('user_id_encoded')
        ]

        self.sequences = []
        self.labels = []

        # Membuat sekuens per user
        for user_data, user_labels in zip(self.data_by_user, self.labels_by_user):
            if len(user_data) >= self.seq_len:
                for i in range(len(user_data) - self.seq_len + 1):
                    self.sequences.append(user_data[i : i + self.seq_len])
                    self.labels.append(user_labels[i + self.seq_len - 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = self.sequences[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# (D) Arsitektur LSTM-AE
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=16):
        super(LSTMAutoencoder, self).__init__()
        self.n_features = n_features
        self.embedding_dim = embedding_dim
        
        # [Image of LSTM Autoencoder architecture diagram]
        self.lstm_encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )
        self.lstm_decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=n_features,
            num_layers=1,
            batch_first=True
        )
        self.decoder_fc = nn.Linear(n_features, n_features)

    def forward(self, x):
        seq_len = x.shape[1]
        _, (hidden_state, _) = self.lstm_encoder(x)
        encoded_seq = hidden_state[-1].unsqueeze(1) 
        decoder_input = encoded_seq.repeat(1, seq_len, 1)
        reconstructed_seq, _ = self.lstm_decoder(decoder_input)
        reconstructed_seq = self.decoder_fc(reconstructed_seq)
        return reconstructed_seq

print("Class Dataset dan Model (Statis & Sekuensial) siap.")


Class Dataset dan Model (Statis & Sekuensial) siap.


In [6]:
# --- Fungsi Helper untuk Training & Evaluasi ---

# Fungsi untuk menghitung error rekonstruksi (STATIS)
def get_statis_errors(model, loader, device):
    model.eval()
    criterion = nn.MSELoss(reduction='none') # reduction='none' agar kita dapat error per sampel
    all_errors = []
    all_labels = []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            reconstructions = model(features)
            # Hitung MSE per baris (dim=1)
            errors = criterion(reconstructions, features).mean(dim=1) 
            all_errors.append(errors.cpu().numpy())
            all_labels.append(labels.numpy())
            
    return np.concatenate(all_errors), np.concatenate(all_labels)

# Fungsi untuk menghitung error rekonstruksi (SEKUENTIAL)
def get_seq_errors(model, loader, device):
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    all_errors = []
    all_labels = []
    with torch.no_grad():
        for seqs, labels in loader:
            seqs = seqs.to(device)
            reconstructions = model(seqs)
            # Hitung MSE per sekuens (dim=1 dan dim=2)
            errors = criterion(reconstructions, seqs).mean(dim=(1, 2)) 
            all_errors.append(errors.cpu().numpy())
            all_labels.append(labels.numpy())
            
    return np.concatenate(all_errors), np.concatenate(all_labels)

# Fungsi untuk optimasi threshold (PERBAIKAN F1-Plot TERMASUK)
def optimize_threshold_f1(errors, labels):
    precision, recall, thresholds = precision_recall_curve(labels, errors)
    
    # --- (PERBAIKAN: 'ValueError: x and y must have same...') ---
    # Hitung F1-score
    f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)
    # Potong elemen terakhir dari f1_scores, precision, recall agar panjangnya = N-1
    f1_scores = f1_scores[:-1]
    precision = precision[:-1]
    recall = recall[:-1]
    # --- (PERBAIKAN SELESAI) ---
    
    # 
    best_f1_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_f1_idx]
    best_threshold = thresholds[best_f1_idx]
    
    return best_threshold, best_f1, (thresholds, f1_scores)

print("Fungsi helper (training & evaluasi) siap.")


Fungsi helper (training & evaluasi) siap.


In [7]:
# --- Baseline 2: Melatih Autoencoder Statis ---

print("\n--- [Baseline 2] Melatih AE Statis (Simulasi V3) ---")

# 1. Siapkan data statis
# Variabel `df_train_normal`, `df_val_model`, `df_test_model` sudah dibuat di Sel 2
N_FEATURES_SIM = (
    df_train_normal.drop(['Class', 'anomaly_type'], axis=1).shape[1]
)
print(f"Jumlah Fitur Statis: {N_FEATURES_SIM}")

train_ds_statis = StaticDataset(df_train_normal) # (Hanya normal untuk latih)
val_ds_statis = StaticDataset(df_val_model) # (CAMPURAN untuk optimasi threshold)
test_ds_statis = StaticDataset(df_test_model)

# Buat DataLoader (PERBAIKAN: num_workers=0)
train_loader_statis = DataLoader(train_ds_statis, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_statis = DataLoader(val_ds_statis, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_statis = DataLoader(test_ds_statis, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader Statis siap.")

# 2. Inisialisasi & Latih Model AE Statis
model_ae_statis = Static_Autoencoder(n_features=N_FEATURES_SIM).to(device)
loss_fn_statis = nn.MSELoss()
optimizer_statis = optim.Adam(model_ae_statis.parameters(), lr=1e-3, weight_decay=1e-5)

print("Melatih AE Statis...")
for epoch in range(N_EPOCHS):
    model_ae_statis.train()
    for (data, _) in train_loader_statis:
        data = data.to(device)
        reconstructions = model_ae_statis(data)
        loss = loss_fn_statis(reconstructions, data)
        optimizer_statis.zero_grad(); loss.backward(); optimizer_statis.step()
    
    if (epoch+1) % 10 == 0 or epoch == (N_EPOCHS-1):
        print(f"AE Statis Epoch {epoch+1}/{N_EPOCHS}")
print("Pelatihan AE Statis selesai.")

# 3. Optimasi Threshold untuk AE Statis
print("\n--- [Optimasi] Mencari Threshold Terbaik (AE Statis) ---")
val_errors_statis, val_labels_statis = get_statis_errors(
    model_ae_statis, val_loader_statis, device
)

best_threshold_statis, best_f1_statis, _ = optimize_threshold_f1(
    val_errors_statis, val_labels_statis
)
print(f"Threshold Terbaik AE Statis: {best_threshold_statis:.6f} (F1: {best_f1_statis:.4f})")

# 4. Evaluasi Final AE Statis (pada Test Set)
print("\n--- [Evaluasi Final] AE Statis (Tabel 2) ---")
test_errors_statis, test_labels_statis = get_statis_errors(
    model_ae_statis, test_loader_statis, device
)
y_pred_statis = [1 if e > best_threshold_statis else 0 for e in test_errors_statis]

print(classification_report(test_labels_statis, y_pred_statis, target_names=['Normal (0)', 'Fraud (1)']))



--- [Baseline 2] Melatih AE Statis (Simulasi V3) ---
Jumlah Fitur Statis: 4
DataLoader Statis siap.
Melatih AE Statis...
AE Statis Epoch 10/30
AE Statis Epoch 20/30
AE Statis Epoch 30/30
Pelatihan AE Statis selesai.

--- [Optimasi] Mencari Threshold Terbaik (AE Statis) ---
Threshold Terbaik AE Statis: 100.242638 (F1: 0.0904)

--- [Evaluasi Final] AE Statis (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       0.98      0.98      0.98     15490
   Fraud (1)       0.06      0.07      0.06       300

    accuracy                           0.96     15790
   macro avg       0.52      0.52      0.52     15790
weighted avg       0.96      0.96      0.96     15790



In [8]:
# --- [Model Utama] Melatih LSTM-Autoencoder ---

print("\n--- [Model Utama] Melatih LSTM-Autoencoder ---")

# 1. Siapkan data sekuensial
N_FEATURES_SEQ = N_FEATURES_SIM # Jumlah fiturnya sama

# Buat Dataset (menggunakan class FraudDatasetSeq)
train_ds_seq = FraudDatasetSeq(
    df=df_train_normal, # Latih HANYA pada data normal
    seq_len=TIME_STEPS
)
val_ds_seq = FraudDatasetSeq(
    df=df_val_model, # Validasi pada data campuran
    seq_len=TIME_STEPS
)
test_ds_seq = FraudDatasetSeq(
    df=df_test_model, # Tes pada data campuran
    seq_len=TIME_STEPS
)

# Buat DataLoader (PERBAIKAN: num_workers=0)
train_loader_seq = DataLoader(train_ds_seq, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_seq = DataLoader(val_ds_seq, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_seq = DataLoader(test_ds_seq, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader Sekuensial siap.")

# 2. Inisialisasi & Latih Model LSTM-AE
model_lstm_ae = LSTMAutoencoder(n_features=N_FEATURES_SEQ).to(device)
optimizer_lstm = optim.Adam(model_lstm_ae.parameters(), lr=1e-3)
criterion_lstm = nn.MSELoss()

print("Melatih LSTM-AE...")
for epoch in range(N_EPOCHS):
    model_lstm_ae.train()
    total_loss = 0
    for seqs, _ in train_loader_seq:
        seqs = seqs.to(device)
        optimizer_lstm.zero_grad()
        reconstructions = model_lstm_ae(seqs)
        loss = criterion_lstm(reconstructions, seqs)
        loss.backward()
        optimizer_lstm.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0 or epoch == (N_EPOCHS-1):
        # (Tambahkan validasi loss opsional)
        print(f"LSTM-AE Epoch {epoch+1}/{N_EPOCHS}")

print("Pelatihan LSTM-AE selesai.")

# 3. Optimasi Threshold untuk LSTM-AE
print("\n--- [Optimasi] Mencari Threshold Terbaik (LSTM-AE) ---")
val_errors_seq, val_labels_seq = get_seq_errors(
    model_lstm_ae, val_loader_seq, device
)

best_threshold_seq, best_f1_seq, _ = optimize_threshold_f1(
    val_errors_seq, val_labels_seq
)
print(f"Threshold Terbaik LSTM-AE: {best_threshold_seq:.6f} (F1: {best_f1_seq:.4f})")

# 4. Evaluasi Final LSTM-AE (pada Test Set)
print("\n--- [Evaluasi Final] LSTM-AE (Tabel 2) ---")
test_errors_seq, test_labels_seq = get_seq_errors(
    model_lstm_ae, test_loader_seq, device
)
y_pred_seq = [1 if e > best_threshold_seq else 0 for e in test_errors_seq]

print(classification_report(test_labels_seq, y_pred_seq, target_names=['Normal (0)', 'Fraud (1)']))



--- [Model Utama] Melatih LSTM-Autoencoder ---
DataLoader Sekuensial siap.
Melatih LSTM-AE...
LSTM-AE Epoch 10/30
LSTM-AE Epoch 20/30
LSTM-AE Epoch 30/30
Pelatihan LSTM-AE selesai.

--- [Optimasi] Mencari Threshold Terbaik (LSTM-AE) ---
Threshold Terbaik LSTM-AE: 9275973.000000 (F1: 0.0000)

--- [Evaluasi Final] LSTM-AE (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       0.00      0.00      0.00      28.0
   Fraud (1)       0.00      0.00      0.00       0.0

    accuracy                           0.00      28.0
   macro avg       0.00      0.00      0.00      28.0
weighted avg       0.00      0.00      0.00      28.0



In [9]:
# --- [Analisis Final] Menguji Performa pada Anomali Sekuensial (Tabel 4) ---
print("\n--- [Analisis Final] Menguji Performa pada Anomali Sekuensial (Tabel 4) ---")

# Kita perlu mendapatkan error & label dari Test Set, TAPI dengan 'anomaly_type'
# Kita harus mendefinisikan ulang Dataset/DataLoader kita untuk Test Set
# agar 'anomaly_type' juga dikembalikan.

# --- Persiapan Ulang Data Tes (dengan anomaly_type) ---

# (A) Versi Statis
class StaticDatasetWithAnomType(Dataset):
    def __init__(self, df):
        self.features = df.drop(['Class', 'anomaly_type'], axis=1, errors='ignore').values
        self.labels = df['Class'].values
        self.anom_types = df['anomaly_type'].values
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.features[idx], dtype=torch.float32), 
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.anom_types[idx] # Kembalikan tipe anomali
        )

test_ds_statis_anom = StaticDatasetWithAnomType(df_test_model)
test_loader_statis_anom = DataLoader(test_ds_statis_anom, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# (B) Versi Sekuensial
class FraudDatasetSeqWithAnomType(Dataset):
    def __init__(self, df, seq_len=10):
        self.df = df
        self.seq_len = seq_len
        
        feature_cols = ['user_id_encoded', 'hour_of_day', 'day_of_week', 'amount_scaled']
        label_col = 'Class'
        anom_type_col = 'anomaly_type'

        self.data_by_user = [
            group[feature_cols].values 
            for _, group in self.df.groupby('user_id_encoded')
        ]
        self.labels_by_user = [
            group[label_col].values
            for _, group in self.df.groupby('user_id_encoded')
        ]
        self.anom_types_by_user = [
            group[anom_type_col].values
            for _, group in self.df.groupby('user_id_encoded')
        ]

        self.sequences = []
        self.labels = []
        self.anom_types = []

        for user_data, user_labels, user_anom_types in zip(self.data_by_user, self.labels_by_user, self.anom_types_by_user):
            if len(user_data) >= self.seq_len:
                for i in range(len(user_data) - self.seq_len + 1):
                    self.sequences.append(user_data[i : i + self.seq_len])
                    self.labels.append(user_labels[i + self.seq_len - 1])
                    self.anom_types.append(user_anom_types[i + self.seq_len - 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.float32), 
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.anom_types[idx] # Kembalikan tipe anomali
        )

test_ds_seq_anom = FraudDatasetSeqWithAnomType(df=df_test_model, seq_len=TIME_STEPS)
test_loader_seq_anom = DataLoader(test_ds_seq_anom, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader khusus untuk 'Tabel 4' siap.")

# --- Evaluasi AE Statis (untuk Tabel 4) ---
model_ae_statis.eval()
all_errors_statis = []
all_labels_statis = []
all_anom_types_statis = []
with torch.no_grad():
    for features, labels, anom_types in test_loader_statis_anom:
        features = features.to(device)
        reconstructions = model_ae_statis(features)
        errors = nn.MSELoss(reduction='none')(reconstructions, features).mean(dim=1)
        
        all_errors_statis.append(errors.cpu().numpy())
        all_labels_statis.append(labels.numpy())
        all_anom_types_statis.append(anom_types.numpy())

all_errors_statis = np.concatenate(all_errors_statis)
all_labels_statis = np.concatenate(all_labels_statis)
all_anom_types_statis = np.concatenate(all_anom_types_statis)

# Buat prediksi berdasarkan threshold
pred_statis = (all_errors_statis > best_threshold_statis).astype(int)

# --- Evaluasi LSTM-AE (untuk Tabel 4) ---
model_lstm_ae.eval()
all_errors_seq = []
all_labels_seq = []
all_anom_types_seq = []
with torch.no_grad():
    for seqs, labels, anom_types in test_loader_seq_anom:
        seqs = seqs.to(device)
        reconstructions = model_lstm_ae(seqs)
        errors = nn.MSELoss(reduction='none')(reconstructions, seqs).mean(dim=(1, 2))
        
        all_errors_seq.append(errors.cpu().numpy())
        all_labels_seq.append(labels.numpy())
        all_anom_types_seq.append(anom_types.numpy())

all_errors_seq = np.concatenate(all_errors_seq)
all_labels_seq = np.concatenate(all_labels_seq)
all_anom_types_seq = np.concatenate(all_anom_types_seq)

# Buat prediksi berdasarkan threshold
pred_seq = (all_errors_seq > best_threshold_seq).astype(int)

# --- Hitung Metrik untuk Tabel 4 ---
# Kita hanya peduli dengan data 'Fraud' (label=1)
def calculate_f1_by_type(y_true, y_pred, anom_types):
    df = pd.DataFrame({
        'true': y_true, 
        'pred': y_pred, 
        'type': anom_types
    })
    
    # Pisahkan data Tipe 1 (Statis) dan Tipe 3 (Sekuensial)
    df_type_1 = df[df['type'].isin([0, 1])] # Normal (0) + Statis (1)
    df_type_3 = df[df['type'].isin([0, 3])] # Normal (0) + Sekuensial (3)
    
    # Hitung F1-Score untuk Tipe 1
    f1_type_1 = f1_score(df_type_1['true'], df_type_1['pred'], pos_label=1)
    
    # Hitung F1-Score untuk Tipe 3
    f1_type_3 = f1_score(df_type_3['true'], df_type_3['pred'], pos_label=1)
    
    return f1_type_1, f1_type_3

# Hitung untuk AE Statis
f1_statis_t1, f1_statis_t3 = calculate_f1_by_type(
    all_labels_statis, pred_statis, all_anom_types_statis
)

# Hitung untuk LSTM-AE
f1_seq_t1, f1_seq_t3 = calculate_f1_by_type(
    all_labels_seq, pred_seq, all_anom_types_seq
)

print("\n--- Hasil untuk Tabel 4 (F1-Score) ---")
print(f"AE Statis | F1 Tipe 1 (Statis): {f1_statis_t1:.4f} | F1 Tipe 3 (Sekuensial): {f1_statis_t3:.4f}")
print(f"LSTM-AE   | F1 Tipe 1 (Statis): {f1_seq_t1:.4f} | F1 Tipe 3 (Sekuensial): {f1_seq_t3:.4f}")

# Siapkan data untuk disimpan ke CSV
tabel_4_data = {
    'Model': ['AE Statis (Optimized)', 'LSTM-AE (Optimized)'],
    'F1-Score (Anomali Statis, Tipe 1)': [f1_statis_t1, f1_seq_t1],
    'F1-Score (Anomali Sekuensial, Tipe 3)': [f1_statis_t3, f1_seq_t3]
}
df_tabel_4 = pd.DataFrame(tabel_4_data)
df_tabel_4.to_csv(os.path.join(MODEL_PATH, 'Tabel_4_Hasil_Simulasi_F1.csv'), index=False)
print("\nData Tabel 4 disimpan ke CSV.")
print(df_tabel_4)

print("\nNotebook 3 V3 (Final) Selesai!")



--- [Analisis Final] Menguji Performa pada Anomali Sekuensial (Tabel 4) ---
DataLoader khusus untuk 'Tabel 4' siap.

--- Hasil untuk Tabel 4 (F1-Score) ---
AE Statis | F1 Tipe 1 (Statis): 0.0208 | F1 Tipe 3 (Sekuensial): 0.0569
LSTM-AE   | F1 Tipe 1 (Statis): 0.0000 | F1 Tipe 3 (Sekuensial): 0.0000

Data Tabel 4 disimpan ke CSV.
                   Model  F1-Score (Anomali Statis, Tipe 1)  \
0  AE Statis (Optimized)                           0.020779   
1    LSTM-AE (Optimized)                           0.000000   

   F1-Score (Anomali Sekuensial, Tipe 3)  
0                               0.056856  
1                               0.000000  

Notebook 3 V3 (Final) Selesai!
